# EfficientNetV2-B0 pretrained trên CIFAR-100

Notebook này là giao diện tương tác của pipeline Python trong `src/cifar_classifier/pretrained/`. Chạy các cell theo thứ tự. Dữ liệu local nằm ở `cifar-100/`; mỗi lượt train ghi checkpoint, history và cấu hình vào một thư mục mới trong `artifacts/pretrained/runs/`.

Lượt chạy đã lưu trong phiên bản notebook cũ (test accuracy 84,95%) thuộc cấu hình trước khi đồng bộ augmentation. Notebook mới không dùng số đó làm kết quả của pipeline hiện tại. Để có kết quả mới, hãy train và đánh giá lại.

Cài dependency từ thư mục gốc project trước khi mở notebook: `python -m pip install -e '.[pretrained]'`. Máy này đã có trọng số ImageNet trong `artifacts/pretrained/cache/`; máy khác cần kết nối mạng ở lần chạy đầu hoặc chỉ định file trọng số local.

## 1. Cấu hình và kiểm tra dữ liệu

Mặc định giống cấu hình 30 epoch trong notebook cũ: chia train/validation 45.000/5.000 bằng seed 42; batch size 32; ảnh 224×224; patience 4. Đặt `SMOKE_TEST=True` bằng biến môi trường `CIFAR_PRETRAINED_SMOKE=1` để kiểm tra nhanh trên vài ảnh mà không tạo kết quả chính thức.

In [ ]:
import os
import sys
import tempfile
from datetime import datetime
from pathlib import Path

from IPython.display import Image, display

PROJECT_DIR = next(
    (candidate.resolve() for candidate in (Path.cwd(), Path.cwd().parent)
     if (candidate / "src" / "cifar_classifier" / "pretrained").is_dir()
     and (candidate / "cifar-100" / "train").is_file()),
    None,
)
if PROJECT_DIR is None:
    raise FileNotFoundError("Không tìm thấy project và cifar-100/train; mở notebook từ thư mục project.")
sys.path.insert(0, str(PROJECT_DIR / "src"))

from cifar_classifier.pretrained.config import EfficientNetConfig
from cifar_classifier.pretrained.data import validate_data_dir
from cifar_classifier.pretrained.engine import choose_device
from cifar_classifier.pretrained.evaluate import evaluate
from cifar_classifier.pretrained.plots import plot_history
from cifar_classifier.pretrained.train import train

DATA_DIR = PROJECT_DIR / "cifar-100"
CACHE_DIR = PROJECT_DIR / "artifacts" / "pretrained" / "cache"
validate_data_dir(DATA_DIR)
SMOKE_TEST = os.environ.get("CIFAR_PRETRAINED_SMOKE") == "1"
config = (
    EfficientNetConfig(epochs=1, batch_size=2, image_size=64)
    if SMOKE_TEST else EfficientNetConfig()
)
run_dir = (
    Path(tempfile.mkdtemp(prefix="efficientnet-smoke-"))
    if SMOKE_TEST else
    PROJECT_DIR / "artifacts" / "pretrained" / "runs" / f"efficientnetv2-b0-{datetime.now():%Y%m%d-%H%M%S-%f}"
)
checkpoint_path = run_dir / "best.pth"
print("Project:", PROJECT_DIR)
print("Data:", DATA_DIR)
print("Device:", choose_device())
print("Run:", run_dir)
print("Smoke test:", SMOKE_TEST)
print("Config:", config)

## 2. Train EfficientNetV2-B0

`timm` khởi tạo backbone bằng trọng số ImageNet và thay đầu phân loại bằng 100 lớp. Ảnh train được augment khi còn 32×32, resize lên 224×224 và chuẩn hóa ImageNet. Mỗi batch chọn MixUp 30%, CutMix 30%, hoặc giữ nguyên 40%. Validation không augment. Checkpoint `best.pth` được chọn theo `val_loss` thấp nhất; train dừng sớm sau 4 epoch không cải thiện.

Cell này chạy đầy đủ tối đa 30 epoch ở chế độ thường. Chế độ smoke chỉ dùng vài ảnh để kiểm tra luồng chạy.

In [ ]:
run = train(
    DATA_DIR,
    run_dir,
    config,
    device_name="auto",
    pretrained=True,
    cache_dir=CACHE_DIR,
    limit_train=4 if SMOKE_TEST else None,
    limit_val=2 if SMOKE_TEST else None,
)
print(f"Checkpoint tốt nhất: {checkpoint_path}")
print(f"Epoch tốt nhất: {run['best_epoch']} | val_loss: {run['best_val_loss']:.4f}")

## 3. Vẽ training history

Biểu đồ đọc lại `history.csv` đã lưu, nên có thể chạy lại cell này mà không train lại. Đường dọc đánh dấu epoch của checkpoint tốt nhất theo validation loss.

In [ ]:
history_path = run_dir / "history.csv"
if not history_path.is_file():
    raise FileNotFoundError(f"Chưa có history: {history_path}. Chạy cell train hoặc đặt run_dir đến một run đã có.")
history_image = plot_history(history_path, run_dir / "training_history.png")
display(Image(filename=str(history_image)))

## 4. Đánh giá trên test set

Cell này nạp `best.pth` với `pretrained=False` rồi dùng trọng số trong checkpoint; không tải lại ImageNet. Nó lưu test metrics, báo cáo precision/recall/F1, confusion matrix và ảnh dự đoán mẫu trong `test-evaluation/`. Test set chỉ dùng ở bước này. Nếu muốn đánh giá một run cũ mà không train lại, đặt `run_dir` và `checkpoint_path` theo run đó rồi chạy cell này.

In [ ]:
if not checkpoint_path.is_file():
    raise FileNotFoundError(f"Chưa có checkpoint: {checkpoint_path}. Chạy cell train hoặc chọn một run đã có.")
output_dir = run_dir / "test-evaluation"
metrics = evaluate(
    checkpoint_path,
    DATA_DIR,
    output_dir,
    device_name="auto",
    limit_test=4 if SMOKE_TEST else None,
)
for key, value in metrics.items():
    print(f"{key}: {value}")
print("Các file kết quả:", output_dir)

## 5. Xem ảnh dự đoán và confusion matrix

In [ ]:
for filename in ("sample_predictions.png", "confusion_matrix.png"):
    image_path = output_dir / filename
    if not image_path.is_file():
        raise FileNotFoundError(image_path)
    print(filename)
    display(Image(filename=str(image_path)))
print("Báo cáo theo lớp:", output_dir / "classification_report.csv")